# Diagnostic 3b: fully frozen CGM+Time predictor + image-only residual (CLIP, dim=4)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [6]:
import json

import numpy as np
import pandas as pd
import torch

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline, FrozenImageOnlyResidualPredictor

MODEL_NAME = 'Diagnostic 3b: frozen CGM+time + clip image-only residual (dim=4)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

ENCODER_NAME = 'clip'
IMG_FEAT_DIM = 768
IMG_PROJ_DIM = 4

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' / '3_diagnostic'
EXP2_CHECKPOINT_DIR = (
    PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
)
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_3b'
SUMMARY_PATH = RESULTS_DIR / 'diagnostic3b_clip_dim4_validation_summary.json'
PAIRED_PATH = RESULTS_DIR / 'diagnostic3b_clip_dim4_paired_validation.csv'

print('MODEL_NAME:', MODEL_NAME)
print('ENCODER_NAME:', ENCODER_NAME)
print('IMG_PROJ_DIM:', IMG_PROJ_DIM)
print('RESULTS_DIR:', RESULTS_DIR)



MODEL_NAME: Diagnostic 3b: frozen CGM+time + clip image-only residual (dim=4)
ENCODER_NAME: clip
IMG_PROJ_DIM: 4
RESULTS_DIR: /content/drive/MyDrive/dissertation/experiments/4_diagnostic/3_diagnostic/results/diagnostic_3b


In [7]:
stats = data.load_cgm_stats()

train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

assert len(val_participant_ids) == len(val_loader.dataset)

sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])

print('number of validation participants:', len(VAL_PARTICIPANT_IDS))
print('validation participant IDs:', VAL_PARTICIPANT_IDS)

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]
number of validation participants: 6
validation participant IDs: [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]


In [8]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    pred = model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device))
    return pred, target.to(device)

In [9]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"\n{'-' * 18} seed={seed} {'-' * 18}")
    base_model = load_frozen_baseline(seed, EXP2_CHECKPOINT_DIR)

    train.set_seed(seed)

    model = FrozenImageOnlyResidualPredictor(
        base_model=base_model,
        img_feat_dim=IMG_FEAT_DIM,
        img_proj_dim=IMG_PROJ_DIM,
    )

    baseline_trainable = [
        name for name, p in model.named_parameters()
        if p.requires_grad and (
            name.startswith('gru.')
            or name.startswith('time_proj.')
            or name.startswith('base_head.')
        )
    ]
    if baseline_trainable:
        raise RuntimeError(
            'Diagnostic 3b baseline is not fully frozen. '
            f'Unexpected trainable parameters: {baseline_trainable}'
        )

    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
    print('trainable parameters:')
    for name in trainable_names:
        print('  ', name)

    check_batch = next(iter(val_loader))
    cgm, tsm, has_meal, img_feat, _ = check_batch

    model.eval()
    base_model.eval()

    with torch.no_grad():
        base_pred = base_model(cgm, tsm, has_meal)
        residual_pred = model(cgm, tsm, img_feat, has_meal)

    init_diff = (base_pred - residual_pred).abs().max().item()
    print(f'initial max |residual - baseline| = {init_diff:.8f}')

    if init_diff > 1e-5:
        raise RuntimeError(
            f'seed={seed}: residual model does not match the frozen baseline '
            f'at initialisation (max diff={init_diff}).'
        )

    model, history = train.train_model(model, train_loader, val_loader,
                                       stats['cgm_mean'], stats['cgm_std'],
                                       lr=1e-4, weight_decay=1e-5, forward_fn=forward_fn,
                                       checkpoint_dir=(EXPERIMENT_DIR / 'checkpoints' / f'diagnostic3b_clip_dim4_seed{seed}'),
                                       verbose=False, progress_bar=True, progress_desc=f'D3b seed={seed}')

    n_epochs_trained = len(history['train_loss'])
    val_loss_hist = history['val_loss']
    best_epoch = int(np.argmin(val_loss_hist) + 1)

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(y_pred_val, y_true_val, val_participant_ids,
                                                                   stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME,
        'seed': seed,
        'encoder': ENCODER_NAME,
        'img_feat_dim': IMG_FEAT_DIM,
        'img_proj_dim': IMG_PROJ_DIM,
        'best_epoch': len(history['train_loss']),
        'n_epochs_trained': n_epochs_trained,
        'initial_max_abs_diff_vs_baseline': init_diff,
        'train_loss_history': [float(x) for x in history['train_loss']],
        'val_loss_history': [float(x) for x in val_loss_hist],
        'val_results': val_results,
        'val_results_per_participant': val_results_per_participant,
    }

    all_results.append(seed_result)

    result_path = RESULTS_DIR / f'diagnostic3b_clip_dim4_seed{seed}_results.json'
    with open(result_path, 'w') as f:
        json.dump(seed_result, f, indent=2)

    print(f'best epoch: {best_epoch}; epochs trained: {n_epochs_trained}')
    for h in HORIZONS:
        entry = val_results.get(str(h), val_results.get(h))
        print(f'  {h:>3}-min validation RMSE = {entry["rmse"]:.4f}')

print(f'\nCompleted Diagnostic 3b for {len(all_results)} seeds.')



------------------ seed=7 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


D3b seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 3; epochs trained: 13
   15-min validation RMSE = 10.5609
   30-min validation RMSE = 16.6187
   60-min validation RMSE = 23.9356
   90-min validation RMSE = 28.2522
  120-min validation RMSE = 31.0305

------------------ seed=14 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


D3b seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 4; epochs trained: 14
   15-min validation RMSE = 10.6117
   30-min validation RMSE = 16.6757
   60-min validation RMSE = 24.0322
   90-min validation RMSE = 28.3184
  120-min validation RMSE = 31.1091

------------------ seed=21 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


...

best epoch: 3; epochs trained: 13
   15-min validation RMSE = 10.5924
   30-min validation RMSE = 16.6673
   60-min validation RMSE = 23.9664
   90-min validation RMSE = 28.2408
  120-min validation RMSE = 30.9957

------------------ seed=63 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


D3b seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 3; epochs trained: 13
   15-min validation RMSE = 10.5573
   30-min validation RMSE = 16.6886
   60-min validation RMSE = 23.9063
   90-min validation RMSE = 28.2022
  120-min validation RMSE = 31.0137

------------------ seed=70 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: FrozenImageOnlyResidualPredictor  params: 1,029,502


D3b seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 3; epochs trained: 13
   15-min validation RMSE = 10.5566
   30-min validation RMSE = 16.6350
   60-min validation RMSE = 23.9102
   90-min validation RMSE = 28.1961
  120-min validation RMSE = 31.0705

Completed Diagnostic 3b for 10 seeds.


In [12]:
def _horizon_entry(result_dict, horizon):
    if str(horizon) in result_dict:
        return result_dict[str(horizon)]
    if horizon in result_dict:
        return result_dict[horizon]
    raise KeyError(
        f'Horizon {horizon} not found. Available keys: {list(result_dict.keys())}'
    )


def _participant_entry(result_dict, pid):
    if str(pid) in result_dict:
        return result_dict[str(pid)]
    if pid in result_dict:
        return result_dict[pid]
    raise KeyError(
        f'Participant {pid} not found. Available keys: {list(result_dict.keys())}'
    )


best_epochs = [r['best_epoch'] for r in all_results]

summary = {}
for h in HORIZONS:
    # mean and sd1: overall validation RMSE across seeds
    seed_values = [
        float(_horizon_entry(r['val_results'], h)['rmse'])
        for r in all_results
    ]
    mean_rmse = float(np.mean(seed_values))
    sd1 = float(np.std(seed_values, ddof=1))

    # sd2: for each participant, first average RMSE across seeds;
    # then calculate SD across validation participants.
    participant_means = []
    for pid in VAL_PARTICIPANT_IDS:
        values_across_seeds = []
        for r in all_results:
            participant_result = _participant_entry(
                r['val_results_per_participant'], pid
            )
            h_result = _horizon_entry(participant_result, h)
            values_across_seeds.append(float(h_result['rmse']))

        participant_means.append(float(np.mean(values_across_seeds)))

    sd2 = float(np.std(participant_means, ddof=1))

    summary[str(h)] = {
        'rmse_mean': mean_rmse,
        'sd1_across_seeds': sd1,
        'sd2_across_participants': sd2,
    }

print(
    '=== Diagnostic 3b: Frozen CGM+Time + Image Residual '
    '(CLIP, img_proj_dim=4, train+val only) ==='
)
print(
    'RMSE is reported as mean +/- sd1 (sd2), where sd1 is across seeds '
    'and sd2 is across validation participants.'
)
print(
    f'Best epoch = {np.mean(best_epochs):.2f} +/- '
    f'{np.std(best_epochs, ddof=1):.2f}'
)

for h in HORIZONS:
    s = summary[str(h)]
    print(
        f'{h:>3}-min: RMSE = {s["rmse_mean"]:.2f} +/- '
        f'{s["sd1_across_seeds"]:.2f} '
        f'({s["sd2_across_participants"]:.2f}) mg/dL'
    )

with open(SUMMARY_PATH, 'w') as f:
    json.dump(
        {
            'model': MODEL_NAME,
            'seeds': SEEDS,
            'n_seeds': len(SEEDS),
            'encoder': ENCODER_NAME,
            'img_proj_dim': IMG_PROJ_DIM,
            'best_epoch': {
                'mean': float(np.mean(best_epochs)),
                'sd': float(np.std(best_epochs, ddof=1)),
            },
            'validation_summary': summary,
        },
        f,
        indent=2,
    )

print(f'\nSaved summary to: {SUMMARY_PATH}')

=== Diagnostic 3b: Frozen CGM+Time + Image Residual (CLIP, img_proj_dim=4, train+val only) ===
RMSE is reported as mean +/- sd1 (sd2), where sd1 is across seeds and sd2 is across validation participants.
Best epoch = 13.00 +/- 0.67
 15-min: RMSE = 10.55 +/- 0.06 (2.08) mg/dL
 30-min: RMSE = 16.61 +/- 0.07 (3.61) mg/dL
 60-min: RMSE = 23.91 +/- 0.07 (7.04) mg/dL
 90-min: RMSE = 28.19 +/- 0.06 (10.24) mg/dL
120-min: RMSE = 30.99 +/- 0.08 (12.09) mg/dL

Saved summary to: /content/drive/MyDrive/dissertation/experiments/4_diagnostic/3_diagnostic/results/diagnostic_3b/diagnostic3b_clip_dim4_validation_summary.json


In [13]:
BASE_DIR = (
    PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
    / 'results' / 'gated_cgm_time'
)
D3A_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_3_clip'
D3A_STUB = 'diagnostic3_partial_freeze'


def load_seed_results(result_dir, stub, seeds):
    loaded = {}
    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'
        if not path.exists():
            raise FileNotFoundError(
                f'Missing result file for seed {seed}: {path}\n'
                'Check result_dir/stub before running this comparison.'
            )
        with open(path) as f:
            result = json.load(f)
        if 'val_results' not in result:
            raise KeyError(
                f'{path.name} does not contain "val_results". '
                f'Available keys: {list(result.keys())}'
            )
        loaded[seed] = result
    return loaded


def rmse_by_seed(loaded_results, seeds, horizons):
    return {
        h: np.asarray(
            [
                float(_horizon_entry(loaded_results[seed]['val_results'], h)['rmse'])
                for seed in seeds
            ],
            dtype=float,
        )
        for h in horizons
    }


def summarise_rmse_with_sd2(loaded_results, seeds, horizons, participant_ids):
    summary_out = {}
    for h in horizons:
        seed_values = [
            float(_horizon_entry(loaded_results[seed]['val_results'], h)['rmse'])
            for seed in seeds
        ]

        participant_means = []
        for pid in participant_ids:
            values_across_seeds = []
            for seed in seeds:
                result = loaded_results[seed]
                if 'val_results_per_participant' not in result:
                    raise KeyError(
                        f'Seed {seed} result does not contain '
                        '"val_results_per_participant".'
                    )
                participant_result = _participant_entry(
                    result['val_results_per_participant'], pid
                )
                h_result = _horizon_entry(participant_result, h)
                values_across_seeds.append(float(h_result['rmse']))

            participant_means.append(float(np.mean(values_across_seeds)))

        summary_out[h] = {
            'mean': float(np.mean(seed_values)),
            'sd1': float(np.std(seed_values, ddof=1)),
            'sd2': float(np.std(participant_means, ddof=1)),
        }

    return summary_out


# Load the seed-matched previous experiments.
base_loaded = load_seed_results(BASE_DIR, 'gated_cgm_time', SEEDS)
d3a_loaded = load_seed_results(D3A_DIR, D3A_STUB, SEEDS)
d3b_loaded = {int(r['seed']): r for r in all_results}

# Safety checks.
current_seeds = [int(r['seed']) for r in all_results]
if current_seeds != SEEDS:
    raise ValueError(
        f'Current Diagnostic 3b seed order does not match SEEDS.\n'
        f'Expected: {SEEDS}\nFound:    {current_seeds}'
    )

for name, loaded in [
    ('CGM+Time', base_loaded),
    ('D3a Partial Freeze', d3a_loaded),
    ('D3b Residual', d3b_loaded),
]:
    missing = [
        seed for seed in SEEDS
        if 'val_results_per_participant' not in loaded[seed]
    ]
    if missing:
        raise KeyError(
            f'{name} is missing val_results_per_participant for seeds: {missing}'
        )

base_by_h = rmse_by_seed(base_loaded, SEEDS, HORIZONS)
d3a_by_h = rmse_by_seed(d3a_loaded, SEEDS, HORIZONS)
d3b_by_h = rmse_by_seed(d3b_loaded, SEEDS, HORIZONS)

base_summary = summarise_rmse_with_sd2(
    base_loaded, SEEDS, HORIZONS, VAL_PARTICIPANT_IDS
)
d3a_summary = summarise_rmse_with_sd2(
    d3a_loaded, SEEDS, HORIZONS, VAL_PARTICIPANT_IDS
)
d3b_summary = summarise_rmse_with_sd2(
    d3b_loaded, SEEDS, HORIZONS, VAL_PARTICIPANT_IDS
)

print('=== Validation RMSE comparison ===')
print(
    'RMSE is reported as mean +/- sd1 (sd2), where sd1 is across seeds '
    'and sd2 is across validation participants.'
)
for h in HORIZONS:
    b = base_summary[h]
    a = d3a_summary[h]
    r = d3b_summary[h]

    print(f'\n{h}-min')
    print(
        f'  CGM+Time:          {b["mean"]:.2f} +/- {b["sd1"]:.2f} '
        f'({b["sd2"]:.2f}) mg/dL'
    )
    print(
        f'  Partial Freeze d=4:{a["mean"]:>6.2f} +/- {a["sd1"]:.2f} '
        f'({a["sd2"]:.2f}) mg/dL'
    )
    print(
        f'  Residual d=4:      {r["mean"]:.2f} +/- {r["sd1"]:.2f} '
        f'({r["sd2"]:.2f}) mg/dL'
    )


def print_paired_effect(title, reference_by_h, candidate_by_h, candidate_label):
    print(f'\n=== {title} ===')
    print(
        f"{'horizon':>9}"
        f"{'delta RMSE':>22}"
        f"{'improvement %':>24}"
        f"{candidate_label:>20}"
    )

    for h in HORIZONS:
        ref = reference_by_h[h]
        cand = candidate_by_h[h]

        # Negative delta = candidate has lower RMSE.
        delta = cand - ref
        # Positive percentage = candidate improvement.
        improvement_pct = (ref - cand) / ref * 100.0

        wins = int(np.sum(cand < ref))
        ties = int(np.sum(np.isclose(cand, ref, atol=1e-12)))
        losses = len(SEEDS) - wins - ties

        print(
            f"{h:>6} min"
            f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<7.4f}"
            f"{np.mean(improvement_pct):>13.3f} +/- "
            f"{np.std(improvement_pct, ddof=1):<7.3f}"
            f"{wins:>8}/{len(SEEDS)}"
            f"  (ties={ties}, losses={losses})"
        )


print_paired_effect(
    'Diagnostic 3b paired effect: Residual vs CGM+Time',
    base_by_h,
    d3b_by_h,
    'D3b better seeds',
)

print_paired_effect(
    'Diagnostic 3b vs Diagnostic 3a: Residual vs Partial Freeze',
    d3a_by_h,
    d3b_by_h,
    'D3b better seeds',
)

# Optional seed-level detail table.
rows = []
for h in HORIZONS:
    for i, seed in enumerate(SEEDS):
        base = base_by_h[h][i]
        d3a = d3a_by_h[h][i]
        d3b = d3b_by_h[h][i]

        rows.append({
            'horizon_min': h,
            'seed': seed,
            'cgm_time_rmse': base,
            'diagnostic3a_partial_freeze_rmse': d3a,
            'diagnostic3b_residual_rmse': d3b,
            'd3b_minus_base': d3b - base,
            'd3b_minus_d3a': d3b - d3a,
            'd3b_improvement_vs_base_pct': (base - d3b) / base * 100.0,
            'd3b_improvement_vs_d3a_pct': (d3a - d3b) / d3a * 100.0,
        })

paired_validation_df = pd.DataFrame(rows)
paired_validation_df.to_csv(PAIRED_PATH, index=False)
print(f'\nSaved paired validation table to: {PAIRED_PATH}')

# Uncomment to inspect every seed.
# display(paired_validation_df.round(4))


=== Validation RMSE comparison ===
RMSE is reported as mean +/- sd1 (sd2), where sd1 is across seeds and sd2 is across validation participants.

15-min
  CGM+Time:          10.58 +/- 0.07 (2.09) mg/dL
  Partial Freeze d=4: 10.57 +/- 0.04 (2.14) mg/dL
  Residual d=4:      10.55 +/- 0.06 (2.08) mg/dL

30-min
  CGM+Time:          16.68 +/- 0.07 (3.61) mg/dL
  Partial Freeze d=4: 16.52 +/- 0.06 (3.54) mg/dL
  Residual d=4:      16.61 +/- 0.07 (3.61) mg/dL

60-min
  CGM+Time:          24.07 +/- 0.06 (7.04) mg/dL
  Partial Freeze d=4: 23.85 +/- 0.09 (6.99) mg/dL
  Residual d=4:      23.91 +/- 0.07 (7.04) mg/dL

90-min
  CGM+Time:          28.40 +/- 0.05 (10.27) mg/dL
  Partial Freeze d=4: 28.38 +/- 0.08 (10.17) mg/dL
  Residual d=4:      28.19 +/- 0.06 (10.24) mg/dL

120-min
  CGM+Time:          31.25 +/- 0.06 (12.14) mg/dL
  Partial Freeze d=4: 31.27 +/- 0.11 (12.04) mg/dL
  Residual d=4:      30.99 +/- 0.08 (12.09) mg/dL

=== Diagnostic 3b paired effect: Residual vs CGM+Time ===
  horizon 